In [1]:
import pandas as pd
import optuna
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import cross_val_score
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import SMOTE

In [2]:
# Load the dataset
df = pd.read_csv(r'C:\Users\khiew\Downloads\FYP Reduced (Secondly) Dataset.csv')

In [3]:
# Drop diseases with less than 50 instances
disease_counts = df['diseases'].value_counts()
valid_diseases = disease_counts[disease_counts >= 1000].index
df = df[df['diseases'].isin(valid_diseases)]

# Assuming that the target variable is 'diseases' and all other variables are input features
X = df.drop('diseases', axis=1)
y = df['diseases']

# Encode the target variable (diseases) if it's a categorical variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)
print("Number of remaining classes in training set:", len(np.unique(y_train)))
print("Number of rows left:", len(df))

Number of remaining classes in training set: 37
Number of rows left: 44748


In [4]:
# Apply SMOTE for class balancing in the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# Print the new class distribution after SMOTE
print("Class distribution after SMOTE:")
print(pd.Series(y_train_resampled).value_counts())
print("Number of remaining classes in training set:", len(np.unique(y_train_resampled)))
# Print the number of rows in the resampled training set
print("Number of rows in the resampled training set:", len(X_train_resampled))

Class distribution after SMOTE:
22    1005
27    1005
28    1005
34    1005
6     1005
24    1005
35    1005
12    1005
0     1005
19    1005
18    1005
30    1005
13    1005
5     1005
32    1005
23    1005
16    1005
10    1005
21    1005
33    1005
1     1005
29    1005
4     1005
20    1005
7     1005
31    1005
2     1005
36    1005
8     1005
15    1005
26    1005
9     1005
3     1005
17    1005
25    1005
11    1005
14    1005
Name: count, dtype: int64
Number of remaining classes in training set: 37
Number of rows in the resampled training set: 37185


In [5]:
# Optuna optimization function
def objective(trial):
    # Define the hyperparameters to tune
    n_estimators = trial.suggest_int('n_estimators', 50, 150)
    max_depth = trial.suggest_int('max_depth', 10, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    max_features = trial.suggest_categorical('max_features', ['sqrt', 'log2', None])

    # Create RandomForestClassifier with hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42
    )

    # Perform cross-validation (using 5-fold by default)
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=5, scoring='accuracy')
    accuracy = score.mean()

    # Log the hyperparameters and accuracy for this trial
    print(f"Trial {trial.number}: n_estimators={n_estimators}, max_depth={max_depth}, "
          f"min_samples_split={min_samples_split}, min_samples_leaf={min_samples_leaf}, "
          f"max_features={max_features}, Accuracy={accuracy:.4f}")

    return accuracy

In [6]:
# Create Optuna study for optimization with persistent storage
study = optuna.create_study(
    direction='maximize',  # Assuming you are maximizing accuracy
    study_name="randomforest_diseases_symptoms_dropextremelymore1000+SMOTE_SecondReduction_study", 
    storage=r"sqlite:///C:/Users/khiew/Downloads/randomforest.db", 
    load_if_exists=True  # Load the study if it already exists, to resume from the last trial
)

# Optimize the study with your objective function, you can adjust the n_trials as needed
study.optimize(objective, n_trials=20)
# Print the best trial and hyperparameters
print("\nBest Trial:")
print(study.best_trial)
print("Best Hyperparameters:")
print(study.best_trial.params)

[I 2025-04-22 19:11:27,912] A new study created in RDB with name: randomforest_diseases_symptoms_dropextremelymore1000+SMOTE_SecondReduction_study
[I 2025-04-22 19:11:36,057] Trial 0 finished with value: 0.4577114427860697 and parameters: {'n_estimators': 117, 'max_depth': 18, 'min_samples_split': 15, 'min_samples_leaf': 19, 'max_features': None}. Best is trial 0 with value: 0.4577114427860697.


Trial 0: n_estimators=117, max_depth=18, min_samples_split=15, min_samples_leaf=19, max_features=None, Accuracy=0.4577


[I 2025-04-22 19:11:39,307] Trial 1 finished with value: 0.46182600510958716 and parameters: {'n_estimators': 82, 'max_depth': 48, 'min_samples_split': 19, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.46182600510958716.


Trial 1: n_estimators=82, max_depth=48, min_samples_split=19, min_samples_leaf=4, max_features=sqrt, Accuracy=0.4618


[I 2025-04-22 19:11:43,931] Trial 2 finished with value: 0.4593518892026355 and parameters: {'n_estimators': 65, 'max_depth': 38, 'min_samples_split': 4, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 1 with value: 0.46182600510958716.


...

Trial 17: n_estimators=92, max_depth=13, min_samples_split=17, min_samples_leaf=10, max_features=sqrt, Accuracy=0.4640


[I 2025-04-22 19:12:56,656] Trial 18 finished with value: 0.4619604679306172 and parameters: {'n_estimators': 90, 'max_depth': 25, 'min_samples_split': 18, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 17 with value: 0.4639774102460669.


Trial 18: n_estimators=90, max_depth=25, min_samples_split=18, min_samples_leaf=10, max_features=sqrt, Accuracy=0.4620


[I 2025-04-22 19:12:58,614] Trial 19 finished with value: 0.463842947425037 and parameters: {'n_estimators': 53, 'max_depth': 13, 'min_samples_split': 17, 'min_samples_leaf': 8, 'max_features': 'sqrt'}. Best is trial 17 with value: 0.4639774102460669.


Trial 19: n_estimators=53, max_depth=13, min_samples_split=17, min_samples_leaf=8, max_features=sqrt, Accuracy=0.4638

Best Trial:
FrozenTrial(number=17, state=TrialState.COMPLETE, values=[0.4639774102460669], datetime_start=datetime.datetime(2025, 4, 22, 19, 12, 49, 904283), datetime_complete=datetime.datetime(2025, 4, 22, 19, 12, 53, 196864), params={'n_estimators': 92, 'max_depth': 13, 'min_samples_split': 17, 'min_samples_leaf': 10, 'max_features': 'sqrt'}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'n_estimators': IntDistribution(high=150, log=False, low=50, step=1), 'max_depth': IntDistribution(high=50, log=False, low=10, step=1), 'min_samples_split': IntDistribution(high=20, log=False, low=2, step=1), 'min_samples_leaf': IntDistribution(high=20, log=False, low=1, step=1), 'max_features': CategoricalDistribution(choices=('sqrt', 'log2', None))}, trial_id=424, value=None)
Best Hyperparameters:
{'n_estimators': 92, 'max_depth': 13, 'min_samples_split': 1